In [2]:
import pandas as pd
import numpy as np
import json

In [5]:
df = pd.read_csv("combined_races.csv")
df['Datetime'] = pd.to_datetime(df['Date'])

df = df.drop(columns=["Jurisdiction", "Unprocessed Ballots"]).drop_duplicates()

unique_dates = sorted(df['Date'].unique())
date_to_batch_id = {date: i + 1 for i, date in enumerate(unique_dates)}
df['Batch_ID'] = df['Date'].map(date_to_batch_id)

df = df.sort_values(by=["Race", "Batch_ID"]).reset_index(drop=True)
df

,Date,R,D,Vote Difference,Margin,Total Votes Cast,Total Unprocessed Ballots*,Race,Datetime,Batch_ID
0,2022-11-16,48289,49195,906,0.009294,97484,655300,AD-40,2022-11-16,3
1,2022-11-17,68571,66119,2452,0.018205,134690,463050,AD-40,2022-11-17,4
2,2022-11-18,71632,69774,1858,0.013139,141406,463050,AD-40,2022-11-18,5
3,2022-11-21,78771,79282,511,0.003233,158053,332550,AD-40,2022-11-21,6
4,2022-11-22,78771,79282,511,0.003233,158053,25250,AD-40,2022-11-22,7
...,...,...,...,...,...,...,...,...,...,...
182,2022-12-05,68219,68264,45,0.000330,136483,100,SD-16,2022-12-05,15
183,2022-12-06,68290,68302,12,0.000088,136592,100,SD-16,2022-12-06,16
184,2022-12-07,68290,68302,12,0.000088,136592,100,SD-16,2022-12-07,17
185,2022-12-08,68290,68302,12,0.000088,136592,100,SD-16,2022-12-08,18


In [6]:
df['New_Total'] = df.groupby('Race')['Total Votes Cast'].diff().fillna(df['Total Votes Cast'])
df['New_D'] = df.groupby('Race')['D'].diff().fillna(df['D'])

df['New_Dem_Prop'] = np.where(
    df['New_Total'] > 0, 
    df['New_D'] / df['New_Total'], 
    np.nan
)

df

,Date,R,D,Vote Difference,Margin,Total Votes Cast,Total Unprocessed Ballots*,Race,Datetime,Batch_ID,New_Total,New_D,New_Dem_Prop
0,2022-11-16,48289,49195,906,0.009294,97484,655300,AD-40,2022-11-16,3,97484.0,49195.0,0.504647
1,2022-11-17,68571,66119,2452,0.018205,134690,463050,AD-40,2022-11-17,4,37206.0,16924.0,0.454873
2,2022-11-18,71632,69774,1858,0.013139,141406,463050,AD-40,2022-11-18,5,6716.0,3655.0,0.544223
3,2022-11-21,78771,79282,511,0.003233,158053,332550,AD-40,2022-11-21,6,16647.0,9508.0,0.571154
4,2022-11-22,78771,79282,511,0.003233,158053,25250,AD-40,2022-11-22,7,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,2022-12-05,68219,68264,45,0.000330,136483,100,SD-16,2022-12-05,15,880.0,585.0,0.664773
183,2022-12-06,68290,68302,12,0.000088,136592,100,SD-16,2022-12-06,16,109.0,38.0,0.348624
184,2022-12-07,68290,68302,12,0.000088,136592,100,SD-16,2022-12-07,17,0.0,0.0,NaN
185,2022-12-08,68290,68302,12,0.000088,136592,100,SD-16,2022-12-08,18,0.0,0.0,NaN


In [12]:
df_before_nov14 = df[df['Datetime'] <= '2022-11-14']

df = df[df['Race'].isin(df_before_nov14['Race'].unique())]

df_before_nov14 = df[df['Datetime'] <= '2022-11-14']

final_totals_before_nov14 = df_before_nov14.groupby('Race').last().reset_index()

final_totals_before_nov14['Dem_Prop_Before_11_14'] = (
    final_totals_before_nov14['D'] / final_totals_before_nov14['Total Votes Cast']
)

summary_df = final_totals_before_nov14[['Race', 'Datetime', 'D', 'Total Votes Cast', 'Dem_Prop_Before_11_14']]
summary_df

,Race,Datetime,D,Total Votes Cast,Dem_Prop_Before_11_14
0,CD-13,2022-11-14,39613,79310,0.499470
1,CD-22,2022-11-14,26799,56476,0.474520
2,CD-27,2022-11-14,61342,137545,0.445978
3,CD-3,2022-11-14,79188,168338,0.470411
4,CD-41,2022-11-14,78918,161902,0.487443
5,CD-45,2022-11-14,76252,164997,0.462142
6,CD-47,2022-11-14,105045,204952,0.512535
7,CD-49,2022-11-14,127206,241904,0.525853


In [13]:
sorted_races = sorted(summary_df['Race'].unique())

race_to_id = {race: i + 1 for i, race in enumerate(sorted_races)}
id_to_race = {i + 1: race for i, race in enumerate(sorted_races)}

summary_df['race_id'] = summary_df['Race'].map(race_to_id)
summary_df

,Race,Datetime,D,Total Votes Cast,Dem_Prop_Before_11_14,race_id
0,CD-13,2022-11-14,39613,79310,0.499470,1
1,CD-22,2022-11-14,26799,56476,0.474520,2
2,CD-27,2022-11-14,61342,137545,0.445978,3
3,CD-3,2022-11-14,79188,168338,0.470411,4
4,CD-41,2022-11-14,78918,161902,0.487443,5
5,CD-45,2022-11-14,76252,164997,0.462142,6
6,CD-47,2022-11-14,105045,204952,0.512535,7
7,CD-49,2022-11-14,127206,241904,0.525853,8


In [17]:
baseline_df = summary_df.copy()
if 'Race' not in baseline_df.columns:
    baseline_df['Race'] = baseline_df['race_id'].map(id_to_race)

baseline_df = baseline_df.rename(columns={
    'D': 'baseline_dem_votes',
    'Total Votes Cast': 'baseline_total_votes',
    'Dem_Prop_Before_11_14': 'baseline_dem_proportion'
})

baseline_df = baseline_df[['Race', 'race_id', 'baseline_dem_votes', 'baseline_total_votes', 'baseline_dem_proportion']]

drops = df[df['Datetime'] > '2022-11-14'].copy()

drops_sorted = drops.sort_values(by=['Race', 'Datetime'])

final_drops = drops_sorted.drop_duplicates(subset=['Race'], keep='last').copy()

final_drops = final_drops.rename(columns={
    'D': 'final_dem_votes',
    'Total Votes Cast': 'final_total_votes'
})

final_drops['final_dem_proportion'] = final_drops['final_dem_votes'] / final_drops['final_total_votes']

final_drops_subset = final_drops[['Race', 'final_dem_votes', 'final_total_votes', 'final_dem_proportion']]

race_summary_df = pd.merge(baseline_df, final_drops_subset, on='Race', how='left')

race_summary_df = race_summary_df.sort_values('race_id').reset_index(drop=True)

race_summary_df.to_csv("race_data.csv", index=False)

In [23]:
drops = df[df['Datetime'] > '2022-11-14'].copy()
drops['Votes_Remaining'] = drops['Total Unprocessed Ballots*']
drops

,Date,R,D,Vote Difference,Margin,Total Votes Cast,Total Unprocessed Ballots*,Race,Datetime,Batch_ID,New_Total,New_D,New_Dem_Prop,Votes_Remaining
46,2022-11-15,50442,51203,761,0.007487,101645,237007,CD-13,2022-11-15,2,22335.0,11590.0,0.518916,237007
47,2022-11-16,55921,56521,600,0.005336,112442,159072,CD-13,2022-11-16,3,10797.0,5318.0,0.492544,159072
48,2022-11-17,60084,59121,963,0.008079,119205,84766,CD-13,2022-11-17,4,6763.0,2600.0,0.384445,84766
49,2022-11-18,62529,61702,827,0.006657,124231,84766,CD-13,2022-11-18,5,5026.0,2581.0,0.513530,84766
50,2022-11-21,63539,62674,865,0.006853,126213,64752,CD-13,2022-11-21,6,1982.0,972.0,0.490414,64752
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,2022-12-05,138089,153370,15281,0.052429,291459,5800,CD-49,2022-12-05,15,0.0,0.0,NaN,5800
173,2022-12-06,138089,153370,15281,0.052429,291459,5800,CD-49,2022-12-06,16,0.0,0.0,NaN,5800
174,2022-12-07,138189,153521,15332,0.052559,291710,4600,CD-49,2022-12-07,17,251.0,151.0,0.601594,4600
175,2022-12-08,138189,153521,15332,0.052559,291710,4600,CD-49,2022-12-08,18,0.0,0.0,NaN,4600


In [24]:
drops = pd.merge(drops, race_summary_df, on='Race', how='left')
drops = drops.rename(columns={
    'New_D': 'this_batch_dem_votes',
    'New_Total': 'this_batch_total_votes',
    'New_Dem_Prop': 'this_batch_dem_proportion',
    'Votes_Remaining': 'this_batch_votes_remaining'
})
drops

,Date,R,D,Vote Difference,Margin,Total Votes Cast,Total Unprocessed Ballots*,Race,Datetime,Batch_ID,...,this_batch_dem_votes,this_batch_dem_proportion,this_batch_votes_remaining,race_id,baseline_dem_votes,baseline_total_votes,baseline_dem_proportion,final_dem_votes,final_total_votes,final_dem_proportion
0,2022-11-15,50442,51203,761,0.007487,101645,237007,CD-13,2022-11-15,2,...,11590.0,0.518916,237007,1,39613,79310,0.499470,66496,133556,0.497889
1,2022-11-16,55921,56521,600,0.005336,112442,159072,CD-13,2022-11-16,3,...,5318.0,0.492544,159072,1,39613,79310,0.499470,66496,133556,0.497889
2,2022-11-17,60084,59121,963,0.008079,119205,84766,CD-13,2022-11-17,4,...,2600.0,0.384445,84766,1,39613,79310,0.499470,66496,133556,0.497889
3,2022-11-18,62529,61702,827,0.006657,124231,84766,CD-13,2022-11-18,5,...,2581.0,0.513530,84766,1,39613,79310,0.499470,66496,133556,0.497889
4,2022-11-21,63539,62674,865,0.006853,126213,64752,CD-13,2022-11-21,6,...,972.0,0.490414,64752,1,39613,79310,0.499470,66496,133556,0.497889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,2022-12-05,138089,153370,15281,0.052429,291459,5800,CD-49,2022-12-05,15,...,0.0,NaN,5800,8,127206,241904,0.525853,153541,291735,0.526303
120,2022-12-06,138089,153370,15281,0.052429,291459,5800,CD-49,2022-12-06,16,...,0.0,NaN,5800,8,127206,241904,0.525853,153541,291735,0.526303
121,2022-12-07,138189,153521,15332,0.052559,291710,4600,CD-49,2022-12-07,17,...,151.0,0.601594,4600,8,127206,241904,0.525853,153541,291735,0.526303
122,2022-12-08,138189,153521,15332,0.052559,291710,4600,CD-49,2022-12-08,18,...,0.0,NaN,4600,8,127206,241904,0.525853,153541,291735,0.526303


In [25]:
drops['up_to_this_batch_dem_votes'] = drops['D']
drops['up_to_this_batch_total_votes'] = drops['Total Votes Cast']
drops['up_to_this_batch_dem_proportion'] = drops['up_to_this_batch_dem_votes'] / drops['up_to_this_batch_total_votes']
drops['up_to_this_batch_no_baseline_dem_votes'] = (
    drops['up_to_this_batch_dem_votes'] - drops['baseline_dem_votes']
)

drops['up_to_this_batch_no_baseline_total_votes'] = (
    drops['up_to_this_batch_total_votes'] - drops['baseline_total_votes']
)
drops['up_to_this_batch_no_baseline_dem_proportion'] = (
    drops['up_to_this_batch_no_baseline_dem_votes'] / drops['up_to_this_batch_no_baseline_total_votes']
)
sorted_batches = sorted(drops['Batch_ID'].unique())

batch_to_id = {batch: i + 1 for i, batch in enumerate(sorted_batches)}
id_to_batch = {i + 1: batch for i, batch in enumerate(sorted_batches)}

drops['batch_id'] = drops['Batch_ID'].map(batch_to_id)

drops["Batch Number"] = drops['Batch_ID']
drops

,Date,R,D,Vote Difference,Margin,Total Votes Cast,Total Unprocessed Ballots*,Race,Datetime,Batch_ID,...,final_total_votes,final_dem_proportion,up_to_this_batch_dem_votes,up_to_this_batch_total_votes,up_to_this_batch_dem_proportion,up_to_this_batch_no_baseline_dem_votes,up_to_this_batch_no_baseline_total_votes,up_to_this_batch_no_baseline_dem_proportion,batch_id,Batch Number
0,2022-11-15,50442,51203,761,0.007487,101645,237007,CD-13,2022-11-15,2,...,133556,0.497889,51203,101645,0.503743,11590,22335,0.518916,1,2
1,2022-11-16,55921,56521,600,0.005336,112442,159072,CD-13,2022-11-16,3,...,133556,0.497889,56521,112442,0.502668,16908,33132,0.510322,2,3
2,2022-11-17,60084,59121,963,0.008079,119205,84766,CD-13,2022-11-17,4,...,133556,0.497889,59121,119205,0.495961,19508,39895,0.488984,3,4
3,2022-11-18,62529,61702,827,0.006657,124231,84766,CD-13,2022-11-18,5,...,133556,0.497889,61702,124231,0.496672,22089,44921,0.491730,4,5
4,2022-11-21,63539,62674,865,0.006853,126213,64752,CD-13,2022-11-21,6,...,133556,0.497889,62674,126213,0.496573,23061,46903,0.491674,5,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,2022-12-05,138089,153370,15281,0.052429,291459,5800,CD-49,2022-12-05,15,...,291735,0.526303,153370,291459,0.526215,26164,49555,0.527979,14,15
120,2022-12-06,138089,153370,15281,0.052429,291459,5800,CD-49,2022-12-06,16,...,291735,0.526303,153370,291459,0.526215,26164,49555,0.527979,15,16
121,2022-12-07,138189,153521,15332,0.052559,291710,4600,CD-49,2022-12-07,17,...,291735,0.526303,153521,291710,0.526280,26315,49806,0.528350,16,17
122,2022-12-08,138189,153521,15332,0.052559,291710,4600,CD-49,2022-12-08,18,...,291735,0.526303,153521,291710,0.526280,26315,49806,0.528350,17,18


In [26]:
drops['Datetime'] = pd.to_datetime(drops['Datetime'])

ordered_cols = [
    'Race',
    'race_id',
    'batch_id',
    'Batch Number',
    'Datetime',
    
    'baseline_dem_votes',
    'baseline_total_votes',
    'baseline_dem_proportion',
    
    'this_batch_dem_votes',
    'this_batch_total_votes',
    'this_batch_dem_proportion',
    
    'up_to_this_batch_dem_votes',
    'up_to_this_batch_total_votes',
    'up_to_this_batch_dem_proportion',
    
    'up_to_this_batch_no_baseline_dem_votes',
    'up_to_this_batch_no_baseline_total_votes',
    'up_to_this_batch_no_baseline_dem_proportion',
    
    'final_dem_votes',
    'final_total_votes',
    'final_dem_proportion'
]

drops = drops[ordered_cols].copy()

In [27]:
sorted_races = sorted(race_summary_df['Race'].unique())
race_to_id = {race: i for i, race in enumerate(sorted_races)}
id_to_race = {i: race for i, race in enumerate(sorted_races)}

race_summary_df['race_id'] = race_summary_df['Race'].map(race_to_id)
drops['race_id'] = drops['Race'].map(race_to_id)

sorted_batches = sorted(drops['Batch Number'].unique())
batch_to_id = {int(batch): i for i, batch in enumerate(sorted_batches)}
id_to_batch = {i: int(batch) for i, batch in enumerate(sorted_batches)}

drops['batch_id'] = drops['Batch Number'].map(batch_to_id)

race_summary_df.to_csv("race_data_2022.csv", index=False)
drops.to_csv("drop_data_2022.csv", index=False)

mapping_dicts = {
    "race_to_id": race_to_id,
    "id_to_race": id_to_race,
    "batch_to_id": batch_to_id,
    "id_to_batch": id_to_batch
}

with open("mapping_dicts_2022.json", "w") as f:
    json.dump(mapping_dicts, f)